# run Trajectory v2 — Fixed Training (Target: 76%+ clean accuracy)

## Changes from v1
- Training: 200 epochs (was 50)
- Learning rate warmup + cosine decay
- Data augmentation: CutOut added
- Weight decay: 5e-4
- Target: 75-80% clean accuracy (matching run original)

## Setup
- Dataset: **rojanregmi1/cifar100-c**
- GPU: T4 x2, Internet ON
- Runtime: ~25 min training + 30 sec tracking

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import json
import copy
import gc
import os
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

NUM_CLASSES = 100
BATCH_SIZE = 64
print(f'NUM_CLASSES: {NUM_CLASSES}, BATCH_SIZE: {BATCH_SIZE}')

In [ ]:
# ================================================================
# CutOut augmentation (standard for CIFAR training)
# ================================================================
class Cutout:
    def __init__(self, n_holes=1, length=16):
        self.n_holes = n_holes
        self.length = length
    def __call__(self, img):
        h, w = img.size(1), img.size(2)
        mask = np.ones((h, w), np.float32)
        for _ in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)
            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)
            mask[y1:y2, x1:x2] = 0.
        mask = torch.from_numpy(mask).unsqueeze(0)
        img = img * mask
        return img

# ================================================================
# Training transforms — stronger augmentation for 75%+
# ================================================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761]),
    Cutout(n_holes=1, length=8),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761]),
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                         download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True,
                         num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                        download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=256, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'Train: {len(trainset)}, Test: {len(testset)}')

In [ ]:
# ================================================================
# Train ResNet-18 — 200 epochs, cosine LR, target 75%+
# ================================================================
model = torchvision.models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

EPOCHS = 200
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f'Training ResNet-18 on CIFAR-100 ({EPOCHS} epochs)...')
print('Modified: conv1=3x3, no maxpool (standard for 32x32 images)')

best_acc = 0
for epoch in range(EPOCHS):
    model.train()
    for x, y in trainloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()
    
    if (epoch + 1) % 20 == 0 or epoch == EPOCHS - 1:
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in testloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
        print(f'  Epoch {epoch+1}/{EPOCHS}: acc={acc:.4f} (best={best_acc:.4f}) lr={scheduler.get_last_lr()[0]:.6f}')

# Use best model
model.load_state_dict(best_state)
model.eval()
SOURCE_STATE = copy.deepcopy(model.state_dict())
print(f'\nTraining done. Best clean accuracy: {best_acc:.4f}')
print(f'Target was 75%+. {"PASSED" if best_acc >= 0.75 else "BELOW TARGET — results may differ from run"}')

In [ ]:
# ================================================================
# Load CIFAR-100-C
# ================================================================
DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'gaussian_noise.npy' in files:
        DATA_PATH = root
        break

if DATA_PATH is None:
    for path in ['/kaggle/input/cifar100-c', '/kaggle/input/cifar100-c/CIFAR-100-C',
                 '/kaggle/input/datasets/rojanregmi1/cifar100-c/CIFAR-100-C']:
        if os.path.exists(path) and 'gaussian_noise.npy' in os.listdir(path):
            DATA_PATH = path
            break

assert DATA_PATH is not None, 'CIFAR-100-C not found! Add dataset: rojanregmi1/cifar100-c'
print(f'CIFAR-100-C at: {DATA_PATH}')

def load_cifar100c(corruption, severity=5, n_samples=5000):
    data = np.load(os.path.join(DATA_PATH, f'{corruption}.npy'))
    labels = np.load(os.path.join(DATA_PATH, 'labels.npy'))
    start = (severity - 1) * 10000
    end = start + n_samples
    X = data[start:end]
    y = labels[start:end]
    X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    mean = torch.tensor([0.5071, 0.4867, 0.4408]).view(1, 3, 1, 1)
    std = torch.tensor([0.2675, 0.2565, 0.2761]).view(1, 3, 1, 1)
    X = (X - mean) / std
    X = X.to(device)
    y = torch.tensor(y, dtype=torch.long).to(device)
    print(f'  Loaded {corruption} sev={severity}: {X.shape}')
    return X, y

# Verify
X_test, y_test = load_cifar100c('gaussian_noise', severity=5, n_samples=200)
model.eval()
with torch.no_grad():
    acc = (model(X_test).argmax(1) == y_test).float().mean().item()
print(f'Source acc on gaussian_noise sev5 (200 samples): {acc:.4f}')
del X_test, y_test; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ================================================================
# Helper functions (identical to run)
# ================================================================

def freeze_except_bn(mdl):
    bn_ids = set()
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                bn_ids.add(id(p))
    for p in mdl.parameters():
        p.requires_grad_(id(p) in bn_ids)

def get_batch_metrics(mdl, xb, yb=None):
    mdl.eval()
    with torch.no_grad():
        logits = mdl(xb)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(1)
        counts = torch.bincount(preds, minlength=NUM_CLASSES).float()
        dom_pct = counts.max().item() / counts.sum().item()
        n_classes = (counts > 0).sum().item()
        h_yx = -(probs * torch.log(probs + 1e-8)).sum(1).mean().item()
        marginal = probs.mean(0)
        h_y = -(marginal * torch.log(marginal + 1e-8)).sum().item()
        acc = (preds == yb).float().mean().item() if yb is not None else None
    result = {
        'dom_pct': round(dom_pct, 4),
        'n_classes': n_classes,
        'h_y': round(h_y, 4),
        'h_yx': round(h_yx, 4),
    }
    if acc is not None:
        result['acc'] = round(acc, 4)
    return result

print('Helpers defined.')

In [ ]:
# ================================================================
# Trajectory Tracker (identical logic to run run_method)
# ================================================================

def track_method(X, y, method, lr=0.05):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    
    if method == 'source':
        trajectory = []
        for i in range(0, X.size(0), BATCH_SIZE):
            xb = X[i:i+BATCH_SIZE]
            yb = y[i:i+BATCH_SIZE]
            if xb.size(0) < 4: continue
            m = get_batch_metrics(model, xb, yb)
            m['batch'] = len(trajectory)
            trajectory.append(m)
        return trajectory, []
    
    model.train()
    freeze_except_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.SGD(params, lr=lr, momentum=0.9)
    
    init_params = None
    current_lambda = 0.5 if method == 'bmia_adaptive' else 1.0
    lambda_history = []
    fisher = None
    TAU = 0.10
    
    if method in ['bmia_adaptive', 'eata']:
        init_params = {pn: p.data.clone() for pn, p in model.named_parameters() if p.requires_grad}
    
    if method == 'eata':
        fisher = {pn: torch.zeros_like(p) for pn, p in model.named_parameters() if p.requires_grad}
        n_fisher = min(128, X.size(0))
        for j in range(0, n_fisher, BATCH_SIZE):
            xb = X[j:j+BATCH_SIZE]
            model.zero_grad()
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            ent = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
            ent.backward()
            for pn, p in model.named_parameters():
                if pn in fisher and p.grad is not None:
                    fisher[pn] += p.grad.data ** 2 * xb.size(0)
        for pn in fisher:
            fisher[pn] /= n_fisher
        model.load_state_dict(copy.deepcopy(SOURCE_STATE))
        model.train()
        freeze_except_bn(model)
        params = [p for p in model.parameters() if p.requires_grad]
        opt = optim.SGD(params, lr=lr, momentum=0.9)
        init_params = {pn: p.data.clone() for pn, p in model.named_parameters() if p.requires_grad}
    
    trajectory = []
    
    for i in range(0, X.size(0), BATCH_SIZE):
        xb = X[i:i+BATCH_SIZE]
        yb = y[i:i+BATCH_SIZE]
        if xb.size(0) < 4: continue
        
        m = get_batch_metrics(model, xb, yb)
        m['batch'] = len(trajectory)
        if method == 'bmia_adaptive':
            m['lambda'] = round(current_lambda, 4)
        trajectory.append(m)
        
        model.train()
        opt.zero_grad()
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        
        if method == 'tent':
            loss = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
        elif method == 'eata':
            ent = -(probs * torch.log(probs + 1e-8)).sum(1)
            reliable = ent < 0.4 * np.log(NUM_CLASSES)
            if reliable.sum() < 2: continue
            ent_loss = ent[reliable].mean()
            fisher_loss = sum((fisher[pn] * (p - init_params[pn]) ** 2).sum()
                             for pn, p in model.named_parameters() if pn in fisher)
            loss = ent_loss + 2000 * fisher_loss
        elif method == 'bmia_adaptive':
            marginal_ent = -(probs.mean(0) * torch.log(probs.mean(0) + 1e-8)).sum()
            cond_ent = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
            mi_loss = -marginal_ent + cond_ent
            anc_loss = sum(((p - init_params[pn]) ** 2).sum()
                          for pn, p in model.named_parameters() if pn in init_params)
            loss = mi_loss + current_lambda * anc_loss
        
        loss.backward()
        opt.step()
        
        if method == 'bmia_adaptive':
            with torch.no_grad():
                batch_preds = model(xb).argmax(1)
                counts = torch.bincount(batch_preds, minlength=NUM_CLASSES).float()
                dom = counts.max().item() / counts.sum().item()
            if dom > TAU:
                current_lambda = min(10.0, current_lambda * 1.1)
            else:
                current_lambda = max(0.01, current_lambda * 0.95)
            lambda_history.append(round(current_lambda, 4))
    
    return trajectory, lambda_history

print('Tracker defined.')

In [ ]:
# ================================================================
# RUN TRACKING
# ================================================================

CORRUPTION = 'gaussian_noise'
SEVERITY = 5
LR = 0.05

X, y = load_cifar100c(CORRUPTION, severity=SEVERITY)
print(f'\nTracking: {CORRUPTION}, sev={SEVERITY}, lr={LR}')
print(f'Samples: {X.shape[0]}, Batches: {X.shape[0] // BATCH_SIZE}')

print('\n--- Source ---')
source_traj, _ = track_method(X, y, 'source', lr=LR)
print(f'  {len(source_traj)} batches')
print(f'  Final dom_pct: {source_traj[-1]["dom_pct"]:.1%}, acc: {source_traj[-1].get("acc", "N/A")}')

gc.collect(); torch.cuda.empty_cache()
print('\n--- TENT ---')
tent_traj, _ = track_method(X, y, 'tent', lr=LR)
print(f'  {len(tent_traj)} batches')
print(f'  Final dom_pct: {tent_traj[-1]["dom_pct"]:.1%}, acc: {tent_traj[-1].get("acc", "N/A")}')

gc.collect(); torch.cuda.empty_cache()
print('\n--- EATA ---')
eata_traj, _ = track_method(X, y, 'eata', lr=LR)
print(f'  {len(eata_traj)} batches')
print(f'  Final dom_pct: {eata_traj[-1]["dom_pct"]:.1%}, acc: {eata_traj[-1].get("acc", "N/A")}')

gc.collect(); torch.cuda.empty_cache()
print('\n--- BMIA Adaptive ---')
bmia_traj, lambda_hist = track_method(X, y, 'bmia_adaptive', lr=LR)
print(f'  {len(bmia_traj)} batches')
print(f'  Final dom_pct: {bmia_traj[-1]["dom_pct"]:.1%}, acc: {bmia_traj[-1].get("acc", "N/A")}')
print(f'  Lambda: {lambda_hist[0]:.4f} -> {lambda_hist[-1]:.4f}')

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
for name, traj in [('Source', source_traj), ('TENT', tent_traj),
                     ('EATA', eata_traj), ('BMIA-A', bmia_traj)]:
    print(f'{name:8s}  dom_pct={traj[-1]["dom_pct"]:.1%}  '
          f'H(Y)={traj[-1]["h_y"]:.2f}  H(Y|X)={traj[-1]["h_yx"]:.2f}  '
          f'acc={traj[-1].get("acc", 0):.1%}  n_cls={traj[-1]["n_classes"]}')
print(f'Lambda trajectory: {lambda_hist[0]:.4f} -> {lambda_hist[-1]:.4f}')

In [ ]:
# ================================================================
# SAVE
# ================================================================

results = {
    'corruption': CORRUPTION,
    'severity': SEVERITY,
    'lr': LR,
    'clean_accuracy': best_acc,
    'source': source_traj,
    'tent': tent_traj,
    'eata': eata_traj,
    'bmia_adaptive': bmia_traj,
    'lambda_history': lambda_hist,
}

with open('V8_trajectory.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved to V8_trajectory.json')
print(f'Clean accuracy: {best_acc:.4f}')
print('DONE.')